# CFG — Classifier-Free Diffusion Guidance (2022)

_Papers · Generative Models_

**Train one diffusion model to be both conditional and unconditional, then at sampling time push it harder toward the condition by a tunable amount — no separate classifier needed.**

---

This notebook builds classifier-free guidance from scratch, one piece at a time. Run each cell, read the note above it, watch the guidance scale pull samples toward the conditioned class, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — PyTorch

### Step 1 — The guided combination, noise schedule, and data

CFG's whole trick is Eq. 6: combine a conditional and an unconditional noise prediction into `et = (1 + w) * ec - w * eu`, where `w` is the guidance scale. We first sanity-check that formula by hand — note how its norm grows as `w` rises, pushing the estimate further along the conditioning direction. Then we set up the standard DDPM forward-noising schedule and a toy two-class 1-D dataset (class 0 near -2, class 1 near +2).

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

# --- Sanity-check the worked example: Eq. 6 guided combination by hand. ---
ec = torch.tensor([0.30, -0.50])      # conditional prediction  eps_theta(z, c)
eu = torch.tensor([0.10, -0.20])      # unconditional prediction eps_theta(z)
for w in (0.0, 1.0, 3.0):
    et = (1 + w) * ec - w * eu        # Eq. 6
    print(f"w={w:.0f}  guided eps = {[round(v, 4) for v in et.tolist()]}  ||.||={et.norm():.4f}")
# w=0 -> (0.3,-0.5); w=1 -> (0.5,-0.8); w=3 -> (0.9,-1.4); norm grows 0.5831 -> 1.6643.


# --- Forward noising schedule (same as paper-ddpm). ---
T = 40
betas  = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar   = torch.cumprod(alphas, 0)                  # alpha-bar_t


# --- Toy 2-class 1-D data: class 0 ~ N(-2, .3^2), class 1 ~ N(+2, .3^2). ---
def sample_data(n):
    c  = torch.randint(0, 2, (n,))
    mu = torch.where(c == 0, torch.tensor(-2.0), torch.tensor(2.0))
    x  = (mu + torch.randn(n) * 0.3).unsqueeze(1)
    return x, c

### Step 2 — A conditional model with a NULL token, trained jointly

One network serves as *both* the conditional and unconditional predictor. The embedding has three rows: class 0, class 1, and a special **NULL** token that means "no condition". During training (Algorithm 1) we randomly replace the true label with NULL with probability `p_uncond`, so the same weights learn `eps_theta(z, c)` and `eps_theta(z)` at once. The loss is the ordinary noise-prediction MSE; only the conditioning input differs.

In [ ]:
NULL = 2                                            # ids 0,1 = classes; id 2 = null token (no condition)

class CondDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Embedding(3, 16)              # 3 rows: class 0, class 1, and NULL
        self.net = nn.Sequential(
            nn.Linear(1 + 1 + 16, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, 1),
        )
    def forward(self, x, t, c):
        te = (t.float() / T).unsqueeze(1)
        return self.net(torch.cat([x, te, self.emb(c)], 1))


# Joint training (Algorithm 1): drop the class with prob p_uncond.
p_uncond = 0.2
net = CondDenoiser()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
for step in range(8000):
    x0, c = sample_data(512)
    drop  = torch.rand(512) < p_uncond              # Alg. 1: with prob p_uncond...
    c_in  = torch.where(drop, torch.full_like(c, NULL), c)   # ...replace label by NULL token
    tb    = torch.randint(0, T, (512,))
    eps   = torch.randn_like(x0)
    ab    = abar[tb].unsqueeze(1)
    xt    = ab.sqrt() * x0 + (1 - ab).sqrt() * eps  # forward-noise (paper-ddpm Eq. 4)
    loss  = ((eps - net(xt, tb, c_in)) ** 2).mean() # ordinary noise-MSE; only c_in differs
    opt.zero_grad()
    loss.backward()
    opt.step()
    if step % 2000 == 0:
        print(f"  step {step:4d}  loss {loss.item():.4f}")

### Step 3 — Guided sampling

At sampling time (Algorithm 2) each reverse step makes **two** predictions from the one network: the conditional `ec` (true class) and the unconditional `eu` (NULL token). We blend them with Eq. 6, then take the usual DDPM denoising step using the guided estimate `e`. Raising `w` extrapolates further past the conditional prediction, steering samples harder toward the class.

In [ ]:
@torch.no_grad()
def sample(n, c_val, w):
    x  = torch.randn(n, 1)                           # z_T ~ N(0, I)
    c  = torch.full((n,), c_val, dtype=torch.long)
    cn = torch.full((n,), NULL,  dtype=torch.long)   # null token for the unconditional pass
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long)
        ec = net(x, tb, c)                           # conditional eps_theta(z, c)
        eu = net(x, tb, cn)                          # unconditional eps_theta(z)
        e  = (1 + w) * ec - w * eu                   # Eq. 6: guided noise estimate
        a, ab = alphas[t], abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)   # paper-ddpm denoising step
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x.squeeze(1)

### Step 4 — Sweep the guidance scale and ablate it

We sample class 1 (true center +2.0) at `w = 0, 1, 3` and watch `|mean - 2.0|` shrink: guidance pulls the cloud toward the class center and tightens it. The final ablation isolates the guidance term directly — `w=0` is the plain conditional model (more diverse, mean further off), `w=3` adds guidance (tighter, more class-typical).

In [ ]:
print("GUIDANCE SWEEP, class 1 (true center +2.0):")
for w in (0.0, 1.0, 3.0):
    s = torch.cat([sample(2000, 1, w) for _ in range(5)])     # 5 batches for a stable estimate
    print(f"  w={w:.0f}:  mean {s.mean():.3f}   std {s.std():.3f}   |mean-2.0| {abs(s.mean().item() - 2.0):.3f}")
# Typical (our small run, not the paper's number): |mean-2.0| shrinks ~0.34 -> ~0.22 -> ~0.14 as w rises
# 0 -> 1 -> 3: guidance pulls samples toward the conditioned class center and sharpens them.

# ABLATION: w=0 (pure conditional) vs w=3 (strong guidance) -- isolates the guidance term.
print("\nABLATION  (w=0 is the plain conditional model; w=3 adds guidance):")
for w in (0.0, 3.0):
    s = torch.cat([sample(2000, 1, w) for _ in range(5)])
    print(f"  w={w:.0f}:  mean {s.mean():.3f}   std {s.std():.3f}")
# w=0 is more diverse but its mean sits further from +2; w=3 is tighter and more class-typical.

## Visualize the data & results

_As we raise the classifier-free guidance scale w, do class-1 samples sharpen and pull toward the true class center (+2.0)?_

### Step 1 — Rebuild the schedule, model, and retrain

This cell is self-contained, so we re-define the noise schedule, the toy data, and the `CondDenoiser` (same architecture, written compactly), then retrain it with the same conditioning-drop trick. This reproduces the trained model from the reference section.

In [ ]:
import torch, torch.nn as nn, math
torch.manual_seed(0)

T = 40
betas  = torch.linspace(1e-4, 0.02, T)
alphas = 1 - betas
abar   = torch.cumprod(alphas, 0)
NULL   = 2                                          # ids 0,1 classes; 2 = null token

def sample_data(n):
    c  = torch.randint(0, 2, (n,))
    mu = torch.where(c == 0, torch.tensor(-2.0), torch.tensor(2.0))
    return (mu + torch.randn(n) * 0.3).unsqueeze(1), c

class CondDenoiser(nn.Module):
    def __init__(self, h=128):
        super().__init__()
        self.emb = nn.Embedding(3, 16)
        self.net = nn.Sequential(
            nn.Linear(18, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, h), nn.SiLU(),
            nn.Linear(h, 1),
        )
    def forward(self, x, t, c):
        return self.net(torch.cat([x, (t.float() / T).unsqueeze(1), self.emb(c)], 1))

net = CondDenoiser()
opt = torch.optim.Adam(net.parameters(), lr=1e-3)
for _ in range(8000):                               # Alg. 1: train with the conditioning drop
    x0, c = sample_data(512)
    c_in  = torch.where(torch.rand(512) < 0.2, torch.full_like(c, NULL), c)
    tb = torch.randint(0, T, (512,))
    eps = torch.randn_like(x0)
    ab = abar[tb].unsqueeze(1)
    xt = ab.sqrt() * x0 + (1 - ab).sqrt() * eps
    loss = ((eps - net(xt, tb, c_in)) ** 2).mean()
    opt.zero_grad()
    loss.backward()
    opt.step()

### Step 2 — Guided sampling and the guidance sweep

We re-define the same Eq. 6 guided sampler, then sweep `w = 0, 1, 3` on class 1. The printed `|mean - 2.0|` shrinks as `w` rises and the spread tightens — the visual confirmation that higher guidance pulls the class-1 cloud toward +2.0 and sharpens it.

In [ ]:
@torch.no_grad()                                    # Alg. 2 / Eq. 6: guided sampling
def sample(n, c_val, w):
    x  = torch.randn(n, 1)
    c  = torch.full((n,), c_val, dtype=torch.long)
    cn = torch.full((n,), NULL,  dtype=torch.long)
    for t in reversed(range(T)):
        tb = torch.full((n,), t, dtype=torch.long)
        ec = net(x, tb, c)
        eu = net(x, tb, cn)
        e  = (1 + w) * ec - w * eu                   # Eq. 6
        a, ab = alphas[t], abar[t]
        mean = (1 / a.sqrt()) * (x - ((1 - a) / (1 - ab).sqrt()) * e)
        if t > 0:
            x = mean + betas[t].sqrt() * torch.randn_like(x)
        else:
            x = mean
    return x.squeeze(1)

for w in (0.0, 1.0, 3.0):                           # class 1, sweep the guidance scale
    s = torch.cat([sample(2000, 1, w) for _ in range(5)])
    print(f"w={w:.0f}  mean {s.mean():.3f}  std {s.std():.3f}  |mean-2.0| {abs(s.mean().item() - 2.0):.3f}")
# Higher w pulls the class-1 cloud toward +2.0 and tightens it. (Our small run, not the paper's number.)

## Practice

Try each one in the empty cell below it, then reveal the worked solution.

**Problem 1.** The ablation: turn guidance off. Your trained model, sampled at $w=3$, produces tight,
            class-typical samples (mean close to the true class center, small spread). Now set $w=0$ and sample
            the same class with everything else identical. What changes, and what does that isolate?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Set the guidance strength to $w=0$ in the Eq. 6 combination; keep the trained weights, schedule, class, and number of steps the same. — _An honest ablation changes exactly one thing &mdash; the guidance term &mdash; so any difference is attributable to guidance, not retraining._
- At $w=0$, $\tilde\epsilon = (1{+}0)\epsilon_\theta(z,c) - 0\cdot\epsilon_\theta(z) = \epsilon_\theta(z,c)$: the sampler reverts to the plain conditional diffusion model. — _The guidance signal $w[\epsilon_\theta(z,c)-\epsilon_\theta(z)]$ is multiplied by zero, so the unconditional prediction no longer influences the step._
- Sample and compare: the $w=0$ samples are more spread out and their mean sits a bit further from the true class center than the $w=3$ samples. — _Guidance is what extrapolates toward the conditioned mode; without it you get the model's honest, more diverse conditional distribution._

**Answer:** At $w=0$ the combination collapses to the plain conditional model $\epsilon_\theta(z,c)$, so
                 the samples are more diverse but less sharply class-typical &mdash; a larger spread and a
                 mean that sits slightly off the true class center. This isolates the guidance term as
                 the thing that trades diversity for fidelity: it is the only piece removed, and it is built
                 purely from the model's own conditional-minus-unconditional difference, no classifier. (The
                 CODE runs exactly this $w=0$ vs large-$w$ comparison.)

</details>

**Problem 2.** Using Eq. 6 with conditional prediction $\epsilon_\theta(z,c) = (0.4, 0.1)$ and unconditional
            prediction $\epsilon_\theta(z) = (0.1, -0.1)$, compute the guided estimate $\tilde\epsilon$ at
            $w = 2$. Show both forms.

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Guidance signal: $\epsilon_\theta(z,c) - \epsilon_\theta(z) = (0.4-0.1,\ 0.1-(-0.1)) = (0.3,\ 0.2)$. — _The bracket in the rearranged Eq. 6 is the conditional-minus-unconditional difference &mdash; the "more like class $c$" direction._
- Signal form: $\tilde\epsilon = \epsilon_\theta(z,c) + w\,[\text{signal}] = (0.4,0.1) + 2\cdot(0.3,0.2) = (0.4,0.1) + (0.6,0.4) = (1.0,\ 0.5)$. — _$\tilde\epsilon = \epsilon_\theta(z,c) + w[\epsilon_\theta(z,c)-\epsilon_\theta(z)]$._
- Weight form (check): $\tilde\epsilon = (1+w)\epsilon_\theta(z,c) - w\,\epsilon_\theta(z) = 3\cdot(0.4,0.1) - 2\cdot(0.1,-0.1) = (1.2,0.3) - (0.2,-0.2) = (1.0,\ 0.5)$. — _Eq. 6 directly; the two forms must agree._

**Answer:** $\tilde\epsilon = (1.0,\ 0.5)$ by both forms. The guided estimate is the conditional
                 prediction $(0.4,0.1)$ pushed two full copies of the guidance signal $(0.3,0.2)$ further along
                 &mdash; a longer, more class-committed correction than the conditional model alone would give.

</details>

**Problem 3.** In CFG, no image classifier is ever trained, yet the method reproduces classifier guidance.
            Where did the classifier go &mdash; what plays its role?

In [ ]:
# Your turn:


<details><summary>Show worked solution</summary>

- Recall classifier guidance steers each step by $\nabla_{z}\log p(c\mid z)$, the gradient of a classifier's log-probability. — _That gradient is the "make it look more like class $c$" direction; classifier guidance needs an actual (noisy-data) classifier to supply it._
- Apply Bayes' rule: $p(c\mid z) \propto p(z\mid c)/p(z)$, so $\nabla_z\log p(c\mid z) = \nabla_z\log p(z\mid c) - \nabla_z\log p(z)$ &mdash; the difference of conditional and unconditional scores. — _The proportionality constant has zero gradient in $z$, so it drops out, leaving only quantities the generative model already estimates._
- Convert scores to noise predictions ($\nabla\log p \propto -\epsilon/\sigma_\lambda$): the classifier direction equals $-\tfrac{1}{\sigma_\lambda}[\epsilon_\theta(z,c) - \epsilon_\theta(z)]$, i.e. conditional-minus-unconditional noise prediction. — _The diffusion model's own two outputs reconstruct the classifier gradient, so the classifier is redundant._

**Answer:** The classifier is replaced by the difference between the model's own conditional and
                 unconditional noise predictions. Bayes' rule turns $p(c\mid z)$ into the ratio
                 $p(z\mid c)/p(z)$, whose log-gradient is the difference of the two scores &mdash; which the
                 diffusion model already provides as $\epsilon_\theta(z,c)-\epsilon_\theta(z)$. This
                 "implicit classifier" is never a real trained network; it is just an algebraic rewrite, which
                 is why CFG is "classifier-free."

</details>